# HW7: Mapping

1. Find a data set with some geographic component. Use the Repository of Data Sources as a start if you don't have one in mind. It will be advantageous if your data set is complete to the extent you will use it (e.g. every state in the United States represented or all countries in Africa shown). Write a short description of your data and what pattern you want to show with it. Make a choropleth map in Altair showing your data. The locale is up to you and your data of choice, but geography must be accurately represented. For example, if showing state-level data, all states must appear. Missing data should be managed in a reasonable way (e.g. showing null value). Completely missing geographical areas (e.g. a hole in your map because of missing data) are not acceptable.

> I found a dataset from the US Census Bureau, which contains population estimates for all states in the United States for each yeat from 2020 to 2025.

> I will calculate the percentage change in population for each state during this time period, to show if there's any migration pattern across the country, and if that has any correlation with the geography location of the states.

In [94]:
import pandas as pd
import altair as alt
from vega_datasets import data

filename = "NST-EST2025-ALLDATA.csv"
pop_df = pd.read_csv(filepath_or_buffer=filename)
pop_df = pop_df[0 < pop_df["STATE"]]
pop_df = pop_df[pop_df["STATE"] < 57]
pop_df.head()

,SUMLEV,REGION,DIVISION,STATE,NAME,ESTIMATESBASE2020,POPESTIMATE2020,POPESTIMATE2021,POPESTIMATE2022,POPESTIMATE2023,...,RDOMESTICMIG2021,RDOMESTICMIG2022,RDOMESTICMIG2023,RDOMESTICMIG2024,RDOMESTICMIG2025,RNETMIG2021,RNETMIG2022,RNETMIG2023,RNETMIG2024,RNETMIG2025
14,40,3,6,1,Alabama,5025437,5032962,5050058,5076868,5117850,...,5.116523,5.474119,5.841064,4.854242,4.510946,5.475145,7.131285,8.317052,9.153863,6.238616
15,40,4,9,2,Alaska,733383,732906,734590,733659,734654,...,-4.147200,-8.049043,-6.692034,-5.686549,-6.140560,-2.957419,-5.357402,-3.511513,-2.141122,-3.482138
16,40,4,8,4,Arizona,7158104,7186647,7274022,7370065,7452073,...,11.649945,9.330046,4.696219,4.571144,4.098354,12.758608,12.973291,10.059412,12.665625,7.816213
17,40,3,7,5,Arkansas,3011530,3014399,3027127,3047429,3069856,...,5.368842,6.118307,5.827749,4.514805,4.661182,5.815087,8.099357,7.656011,8.810666,6.422610
18,40,4,9,6,California,39555703,39527808,39152927,39125347,39181667,...,-11.966309,-8.792478,-8.783147,-6.152055,-5.820040,-10.844764,-3.320973,-1.322436,1.811667,-3.043671


In [95]:
pop_df["5YearPopChangePct"] = (pop_df["POPESTIMATE2025"] - pop_df["POPESTIMATE2020"]) / pop_df["POPESTIMATE2020"] * 100
pop_df.head()

,SUMLEV,REGION,DIVISION,STATE,NAME,ESTIMATESBASE2020,POPESTIMATE2020,POPESTIMATE2021,POPESTIMATE2022,POPESTIMATE2023,...,RDOMESTICMIG2022,RDOMESTICMIG2023,RDOMESTICMIG2024,RDOMESTICMIG2025,RNETMIG2021,RNETMIG2022,RNETMIG2023,RNETMIG2024,RNETMIG2025,5YearPopChangePct
14,40,3,6,1,Alabama,5025437,5032962,5050058,5076868,5117850,...,5.474119,5.841064,4.854242,4.510946,5.475145,7.131285,8.317052,9.153863,6.238616,3.181546
15,40,4,9,2,Alaska,733383,732906,734590,733659,734654,...,-8.049043,-6.692034,-5.686549,-6.140560,-2.957419,-5.357402,-3.511513,-2.141122,-3.482138,0.595438
16,40,4,8,4,Arizona,7158104,7186647,7274022,7370065,7452073,...,9.330046,4.696219,4.571144,4.098354,12.758608,12.973291,10.059412,12.665625,7.816213,6.083101
17,40,3,7,5,Arkansas,3011530,3014399,3027127,3047429,3069856,...,6.118307,5.827749,4.514805,4.661182,5.815087,8.099357,7.656011,8.810666,6.422610,3.330415
18,40,4,9,6,California,39555703,39527808,39152927,39125347,39181667,...,-8.792478,-8.783147,-6.152055,-5.820040,-10.844764,-3.320973,-1.322436,1.811667,-3.043671,-0.436399


In [96]:
states = alt.topo_feature(data.us_10m.url, "states")

alt.Chart(states).mark_geoshape().encode(
    alt.Color("5YearPopChangePct:Q", bin=alt.BinParams(maxbins=10)),
    tooltip=["NAME:N", "5YearPopChangePct:Q"]
).transform_lookup(
    lookup="id",
    from_=alt.LookupData(pop_df, "STATE", ["5YearPopChangePct", "NAME"]),
).project(
    type="albersUsa"
).properties(
    title="Percentage Change in Population from 2020 to 2025 by State",
    width=600,
    height=400
)

alt.Chart(...)

2. Consider the plot you made in question 1. Does it really need to be a map? Justify your reasoning. Then, attempt to show the same or similar data but in a different (line, scatter, bar, etc) plot. What is gained with this view? What is lost?

> No, it doesn't necessarily need to be a map. The percentage change in population can be shown in a bar chart as well. Now the exact percentage change in population for each state can be easier to read that no longer limited by only few color bins.

> However, there are patterns more clearly shown in the map, such as a region generally have similar population change, and states with negative or little population change are likely neighbored with another state with significant population increase. While these information can still being found in the bar chart, it requires more translation from text to geography, especially nearby states might not be next to each other in the bar chart.

In [97]:
alt.Chart(pop_df).mark_bar().encode(
    y=alt.Y("NAME:N", sort="-x"),
    x="5YearPopChangePct:Q",
    tooltip=["NAME:N", "5YearPopChangePct:Q"]
).properties(
    title="Percentage Change in Population from 2020 to 2025 by State",
    width=600,
    height=400
)

alt.Chart(...)

3. Make a point map in Altair of a region external to Seattle and of importance to you, like your home state or a place you've visited. You can find the base geographic data online (best for cities) or use the geographic base maps provided by Altair, filtering the data itself or adjusting the projection to the appropriate region (best for entire counties, states, countries). You are welcome to show any data you like-- you may use the same data you did for plot 1, if appropriate, call new data from OpenStreetMap, or simply plot places of personal importance to you. You should include at least 10 points.

In [98]:
import requests

resp_json = requests.get("https://api.bart.gov/api/stn.aspx?cmd=stns&key=MW9S-E7SL-26DU-VV8V&json=y").json()
stations_df = pd.DataFrame(resp_json["root"]["stations"]["station"])
stations_df.head()

,name,abbr,gtfs_latitude,gtfs_longitude,address,city,county,state,zipcode
0,12th St. Oakland City Center,12TH,37.803768,-122.271450,1245 Broadway,Oakland,alameda,CA,94612
1,16th St. Mission,16TH,37.765062,-122.419694,2000 Mission Street,San Francisco,sanfrancisco,CA,94110
2,19th St. Oakland,19TH,37.808350,-122.268602,1900 Broadway,Oakland,alameda,CA,94612
3,24th St. Mission,24TH,37.752470,-122.418143,2800 Mission Street,San Francisco,sanfrancisco,CA,94110
4,Antioch,ANTC,37.995388,-121.780420,1600 Slatten Ranch Road,Antioch,Contra Costa,CA,94509


In [99]:
stations_df["latitude"] = stations_df["gtfs_latitude"].astype(float)
stations_df["longitude"] = stations_df["gtfs_longitude"].astype(float)
stations_df.head()

,name,abbr,gtfs_latitude,gtfs_longitude,address,city,county,state,zipcode,latitude,longitude
0,12th St. Oakland City Center,12TH,37.803768,-122.271450,1245 Broadway,Oakland,alameda,CA,94612,37.803768,-122.271450
1,16th St. Mission,16TH,37.765062,-122.419694,2000 Mission Street,San Francisco,sanfrancisco,CA,94110,37.765062,-122.419694
2,19th St. Oakland,19TH,37.808350,-122.268602,1900 Broadway,Oakland,alameda,CA,94612,37.808350,-122.268602
3,24th St. Mission,24TH,37.752470,-122.418143,2800 Mission Street,San Francisco,sanfrancisco,CA,94110,37.752470,-122.418143
4,Antioch,ANTC,37.995388,-121.780420,1600 Slatten Ranch Road,Antioch,Contra Costa,CA,94509,37.995388,-121.780420


In [100]:
import math

ba_center_lon = stations_df["longitude"].mean()
ba_center_lat = stations_df["latitude"].mean()
ba_lon_range = stations_df["longitude"].max() - stations_df["longitude"].min()
ba_lat_range = stations_df["latitude"].max() - stations_df["latitude"].min()

map_size = 600
padding = 0.5
scale_alt = (map_size / (max(ba_lon_range, ba_lat_range) + padding)) / (math.pi / 180)

california = alt.topo_feature(data.us_10m.url, "states")

ba_base_alt = alt.Chart(california).mark_geoshape(
    fill="lightgray",
    stroke="white"
).transform_filter(
    (alt.datum.id == 6)
)

ba_points_alt = alt.Chart(stations_df).mark_circle(
    size=150,
    color="#FF0000",
    opacity=0.8
).encode(
    longitude="longitude:Q",
    latitude="latitude:Q",
    tooltip=["name:N", "address:N", "city:N"]
)

(ba_base_alt + ba_points_alt).project(
    type="mercator",
    scale=scale_alt,
    center=[ba_center_lon, ba_center_lat]
).properties(
    width=map_size,
    height=map_size,
    title="BART (Bay Area Rapid Transit) Stations in the San Francisco Bay Area"
)

alt.LayerChart(...)

4. Make a point map in Folium of the same region.  Your map should include styled icons to fit the places you show, and should also include filtering.

In [101]:
import folium

county_name_map = {
    "alameda": "Alameda",
    "contracosta": "Contra Costa",
    "sanfrancisco": "San Francisco",
    "sanmateo": "San Mateo",
    "Contra Costa": "Contra Costa",
    "Santa Clara": "Santa Clara",
}
county_colors = {
    "Alameda": "blue",
    "Contra Costa": "green",
    "San Francisco": "red",
    "San Mateo": "purple",
    "Santa Clara": "orange",
}
stations_df["county_display"] = stations_df["county"].map(
    lambda c: county_name_map.get(c, c.title())
)

map_folium = folium.Map(width=800, height=800,
                        location=[ba_center_lat, ba_center_lon],
                        tiles='cartodbpositron',
                        zoom_start=10, control_scale=True)

county_groups = {}
for county in sorted(stations_df["county_display"].unique()):
    fg = folium.FeatureGroup(name=county, show=True)
    fg.add_to(map_folium)
    county_groups[county] = fg

for idx, row in stations_df.iterrows():
    # Get lat and lon of points
    lon = row['longitude']
    lat = row['latitude']
    county = row['county_display']

    folium.Marker(
        location=[lat, lon],
        icon=folium.Icon(
            color=county_colors.get(county, "gray"),
            icon="train",
            prefix="fa"
        ),
        tooltip=row['name']
    ).add_to(county_groups[county])

folium.LayerControl(collapsed=False).add_to(map_folium)

map_folium